---
toc: false
title: Why Causal Inference?
date: "06/20/2026"
description: Correlation is not causation, except when it is.
categories: causal_inference
image: /enso-thumbnail.jpg
image-alt: The enso circle
---

Well, hello there, dear reader! 
Once again it's been a while since I posted here, and we've got a lot to talk about.
Today I'm catching you up on something I've been off woodshedding for like 2 and a half years.
That's right, today we're talking *Causal Inference*.

## Why causal inference 

### The Purpose of Data Science

Every once in a while my mom asks me if I can re-explain to her what data scientists do. It's a great question really—we in the field should probably ask ourselves more often. In a single sentence I'd say: the purpose of data science is to create value from data.  And what's valuable to the organizations that employ us?  Well an organization has goals, and in order to achieve those goals it needs to make decisions about which actions to take. At its most effective, data science is about decision intelligence—informing a decision making process to help achieve desired outcomes. These decisions come in many forms, from executive deliberation on bold strategic bets to adaptive transaction-level automation occurring millions of times per day.

Examples include
What price to charge for this hotel room on these dates?
Whether to give customers a promotional offer?
How to allocate the quarterly marketing budget?

Behind each of these decisions is a set of what if questions of the form: What would happen if I did this? What would happen if I did that? To inform the decision we can predict answers to these what if questions. 

Often these answers are already valuable, but sometimes we can go one step further and recommend the action corresponding to the most favorable outcome. Thus our business problems can often be formulated as mathematical optimization problems of the form

$$ \underset{x}{\operatorname{argmax}} f(x) $$

where $x$ indexes the possible actions we can take and $f$ is a "what if" function that predicts the benefit of an action. The above expression says: "find the action that maximizes the predicted benefit."

So far so good, but where does that "what if" function come from? 
If only there was an easy and foolproof way to make predictions...


### Prediction

When I arrived in silicon valley in 2017, data science was having a moment. It was still riding high from that 2012 HBS article proclaiming data science as 
[The Sexiest Job of the 21st Century](https://hbr.org/2012/10/data-scientist-the-sexiest-job-of-the-21st-century).
At the time PhDs from fields across academia were flocking to Silicon Valley to join the ranks of the rapidly growing tech companies who, after reading that HBS article, were ready to pay top dollar for people who knew how to coax treasure out of their mountains of messy data.
There were of course many ways to coax out that treasure, but it soon became clear that predictive modeling via machine learning was particularly well suited to solve a wide variety of business problems.
And so, ML was hot.

To the delight of many ex-academics, we found that ML was just statistical modeling—which we had already been using to do scientific research—but with all the emphasis on  prediction accuracy rather than getting insights about the phenomena under study. 
We traded in our residual plots for evval metrics on a holdout set, and boy howdy did it work! 
It's sort of mind blowing how little effort is required to achieve amazing prediction accuracy on lots of different problems, using essentially the same approach for every problem. 
Heck, I've shown you on this very blog how to solve pretty much any tabular data prediction problem to worldclass accuracy over a handfull of posts about how to fit and tune XGBoost models.
Implementing these solutions has more or less reduced down to something like:

```python
from ml_library import Model
from data import X, y, X_new

model = Model()
model.fit(X, y)
predictions = model.predict(X_new)
```

With this amazingly powerful prediction tool in hand, it's no surprise that pretty soon, everything starts to look like a prediction problem. 
Need to choose what product to recommend? Just use customer behavior data to predict what product the customer is most likely to buy.
Need to set the price of a hotel room? Just use historical pricing data to predict the price.
Need to decide whether to offer a promo? Just predict your profit with or without the promo.

![](hammer.jpg){width="50%"}

> "I suppose it is tempting, if the only tool you have is a hammer, to treat everything as if it were a nail."
Abraham Maslow

Effectively we're taking that decision optimization problem $ \underset{x}{\operatorname{argmax}} f(x) $
and saying: well I don't actually know the objective function $f$, but I can estimate it by training an ML model on some historical data, 
then I can use the model to find the best decision.

But often these decision optimization problems weren't actually nails; they were screws. And when we tried to solve them naively with ML, we got screwed.

###  A Cautionary Tale


Let's walk through a concrete example to see what can go wrong when we misuse predictive models for decision making.

Suppose there's an e-commerce firm that's interested in using promotional offers to increase customer retention.
They've got a historical customer-level dataset that tells whether each customer was given an offer and whether they were retained.
The data science team plans to build a model to predict customer retention from promo status and then use that to decide whether to do another round of promos.
We'll simulate the data for this example so we know how everything works.
In our simulation, the promos will have a small positive effect on retention, just as the DS team might expect.
But not all customers are equally likely to churn, and unbeknownst to the DS team, those promo offers aren't given out randomly; actually the sales team manually goes through their customer list and targets those they think are at the highest risk of churning.


In [ ]:
#| code-fold: true
import pandas as pd
import numpy as np

# For reproducibility
np.random.seed(42)
n = 10000

# 1. Simulate risk (continuous variable)
risk = np.random.normal(0, 1, n)

# 2. Simulate promo based on risk (higher risk much more likely to get promo)
promo_prob = 1 / (1 + np.exp(-1.5 * risk))
promo = np.random.binomial(1, promo_prob)

# 3. Simulate retention (higher risk less likely to retain, promo helps slightly)
retention_logit = 1.0 - 2.0 * risk + 0.5 * promo
retention_prob = 1 / (1 + np.exp(-retention_logit))
retention = np.random.binomial(1, retention_prob)

df = pd.DataFrame({
    'customer_id': np.arange(1, n + 1),
    # 'risk': risk,
    'promo': promo,
    'retention': retention
})

# Display first few rows
df.head()

In [ ]:
mean_retention = df.groupby('promo')['retention'].mean()
print(f"Average retention without promo: {mean_retention[0]:.2%}")
print(f"Average retention with promo: {mean_retention[1]:.2%}")

When the DS team checked the retention rates for customers with and without promo offers, they were surprised to find that retention was actually. lower for those who had received the promos.
They mused: "that's odd; our promos must be pretty terrible or something."
Then they proceeded intrepidly, building a model to predict retention using promo status.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Train-test split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Features and target
X_train = train_df[['promo']]
y_train = train_df['retention']
X_test = test_df[['promo']]
y_test = test_df['retention']

# Fit a Random Forest model on train data
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)

# Evaluate on holdout set
holdout_predictions = clf.predict(X_test)
accuracy = accuracy_score(y_test, holdout_predictions)
print(f"Holdout accuracy: {accuracy:.2%}")

# Predict retention for with and without promo
promo_scenarios = pd.DataFrame({'promo': [0, 1]})
predictions = clf.predict_proba(promo_scenarios)[:, 1]

print(f"Predicted retention without promo: {predictions[0]:.2%}")
print(f"Predicted retention with promo: {predictions[1]:.2%}")

# Choose the optimal action (argmax)
best_promo_idx = np.argmax(predictions)
best_promo_val = promo_scenarios.loc[best_promo_idx, 'promo']
print(f"Optimal decision: promo = {best_promo_val}")

 Using their model, they found that  `promo = 0`, no promotional offer, gives the highest retention prediction and is therefore the  optimal decision for the firm.

Womp womp.

---

So what went wrong?
It turns out that our intrepid DS team has fallen victim to the age old saying — 

> *correlation does not imply causation*.

Predictive models are very good at answering  *observational* questions — questions about what sorts of patterns have been typically observed in the past.
Specifically, supervised ML models find the *observational* conditional expectation function:

$$ \hat{f}(x) = E[Y|X=x] $$

which tells us what the response $Y$ tends to look like in training data when feature $X$ happens to take value $x$. In other words, it lives in the realm of correlations.
The DS team's model learned this observational conditional expectation function, which showed that when promos were applied, retention tended to be low; this much was correct.


But when our ill-fated DS team tried to use a predictive model to answer a what if question about promos and their effect on retention, they were actually asking an *interventional* question — a question about what happens when we decide to intervene on a system in a particular way.
The function they actually wanted to know is the *interventional* conditional expectation function:

$$ \hat{f}(x) = E[Y|do(X=x)] $$

which tells us what the response $Y$ tends to look like when some intervention, represented by the *do operator*,  forcibly sets $X$ to value $x$.
This is what the DS team actually wanted — a function that gives the expected retention when promo is set rather than observed.
This expresses a very different situation from the observational distribution.
Consider how promo arises in the observational data; because the sales team was targeting customers with high churn risk, promo tends to occur in cases with low retention, even though it has a slight positive effect on retention.
In contrast, when promo arises from an intervention, promo is no longer associated with high churn risk, and thus we're able to see its positive effect on retention.

So how do we estimate this interventional expectation function?
In general the observational and interventional conditional expectation functions are not the same, and they are only the same when certain special conditions are satisfied.
It  is the function of causal inference  to figure out how to satisfy those conditions so that we can use observational data to answer causal questions.
In other words:

> "Correlation does not imply causation — except when it does.*"


They had assumed that their model would correctly pick up the *causal effect* of promo on retention, but in fact, all it did was capture the *correlation* between them.





## References 

* [Causal Inference for the Brave and True](https://matheusfacure.github.io/python-causality-handbook/landing-page.html)
* Robert's Course 
* Book of Why
* Statistical Rethinking
* Robert's book